In [ ]:
#Try to add legend to pitch plots
#Do logistic regression for set pieces and freekicks

#imported libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from mplsoccer import Pitch
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.metrics import log_loss, roc_auc_score ,classification_report
from sklearn.calibration import CalibrationDisplay

#code below imports and cleans dataset, removing unimportant columns and data without X and Y information
#it also changes result column from strings to integers and removes own goal data
shots = pd.read_csv('shots_dataset.csv')
shots = shots.sort_values('X')
shots = shots.drop(shots.columns[[0,7,12,13,14,15,16,17,18,19]],axis = 1)
shots = shots[~shots.result.str.contains('OwnGoal')]
shots = shots.iloc[3:]
outcome = {'BlockedShot': 0,'SavedShot':0,'MissedShots':0,'ShotOnPost':0,'Goal':1}
shots.result = [outcome[item] for item in shots.result]
shots = shots.reset_index(drop = True)

#code below calculates the distance of the shot from the centre of the goal
shots['deltaX'] = (1 - shots['X'])*105
shots['deltaY'] = abs(shots['Y'] - 0.5)*68
shots['goalDistance'] = ((shots['deltaX']**2) + (shots['deltaY']**2))**(1/2)

#code below calculates the angle between the shot position and the goal posts
numerator = 7.32 * shots['deltaX']
denominator = (shots['deltaX']**2) + (shots['deltaY']**2) - (3.66**2)
angle_rad = np.arctan2(numerator,denominator)
shots['shotAngle'] = np.degrees(angle_rad)

#code below separates open play, set pieces and penalty kicks into different tables
shotsOpenPlay = shots[shots['situation'].str.contains('OpenPlay')]
shotsSetPiece = shots[shots['situation'].str.contains('SetPiece','FromCorner')]
shotsPenalty = shots[shots['situation'].str.contains('Penalty')]
shotsDirectFreekick = shots[shots['situation'].str.contains('DirectFreekick')]

#code below draws pitch and displays location of each open play shot in dataset
pitch = Pitch(pitch_type='metricasports',pitch_length = 105, pitch_width = 68)
fig,ax = pitch.draw()
plt.title('Shots from Open Play')
ax.scatter(shotsOpenPlay['X'],shotsOpenPlay['Y'],c = shotsOpenPlay['result'],s=1)

#code below draws pitch and displays location of each shot after a set piece in dataset
pitch = Pitch(pitch_type='metricasports',pitch_length = 105, pitch_width = 68)
fig,ax = pitch.draw()
plt.title('Shots after Set Pieces')
ax.scatter(shotsSetPiece['X'],shotsSetPiece['Y'],c = shotsSetPiece['result'],s=1)

#code below draws pitch and displays location of each shot on goal from a freekick in dataset
pitch = Pitch(pitch_type='metricasports',pitch_length = 105, pitch_width = 68)
fig,ax = pitch.draw()
plt.title('Shots from Freekicks')
ax.scatter(shotsDirectFreekick['X'],shotsDirectFreekick['Y'],c = shotsDirectFreekick['result'],s=1)
plt.show()

#code below defines features and test statistic
featureColumns = ['goalDistance', 'shotAngle']
xOpen = shotsOpenPlay[featureColumns]
yOpen = shotsOpenPlay.result

#code below trains against dataset
xOpenTrain, xOpenTest, yOpenTrain, yOpenTest = train_test_split(xOpen,yOpen,test_size = 0.25,random_state = 42)
logReg = LogisticRegression(random_state = 42)
logReg.fit(xOpenTrain,yOpenTrain)
yOpenPred = logReg.predict(xOpenTest)

#roc graph (to be labelled and analysied)
yOpenPredProba = logReg.predict_proba(xOpenTest)[::,1]
fpr, tpr, _ = metrics.roc_curve(yOpenTest,  yOpenPredProba)
auc = metrics.roc_auc_score(yOpenTest, yOpenPredProba)
plt.plot(fpr,tpr,label="data 1, auc="+str(auc))
plt.show()

#calculed accuracy (to be analysied)
targetNames = ['No Goal', 'Goal']
print(classification_report(yOpenTest, yOpenPred, target_names=targetNames))

#code plots the predicted probability against distance from goal
plt.scatter(xOpenTest['goalDistance'],yOpenPredProba,s = 1)
plt.ylabel('Predicted Probability')
plt.xlabel('Distance from Goal')
plt.show()

#code plots the predicted probability against shot angle
plt.scatter(xOpenTest['shotAngle'],yOpenPredProba,s = 1)
plt.ylabel('Predicted Probabilty')
plt.xlabel('Shot Angle')
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
disp = CalibrationDisplay.from_predictions(yOpenTest, yOpenPredProba, n_bins=10, strategy="uniform", name="Logistic Regression xG", ax=ax)
ax.set_title("xG Calibration Curve")
ax.set_xlabel("Mean Predicted xG")
ax.set_ylabel("Actual Goal Conversion Rate")
plt.show()

#shotsOpenPlay.head()
